### Descriptors

***Descriptor*** is something more than a ***property***.

A ***descriptor*** is any object that has the `__get__` , `__set__` , or `__delete__` methods defined(***descriptor protocol***).

***Descriptors allow you to CREATE OBJECTS with UNUSUAL BEHAVIOR when ACCESSED as ATTRIBUTES of other objects.***

A ***descriptor*** that implements only the `__get__` method is called a ***non-data descriptor***, and a descriptor that implements the `__set__` and / or `__delete__` methods is called a ***data descriptor***.

In [1]:
class Two:
    def __get__(self, instance, owner):
        print("Get called")
        return 2
    
class Example:
    x = 5  # attribute
    y = Two()  # Descriptor instance
    
ex = Example()

In [4]:
ex.x

5

In [5]:
ex.y

Get called


2

Dynamic Lookups

In [21]:
class ArraySize:
    def __get__(self, instance, owner):  # takes instance and its class reference
        return len(instance.arr)  # return instance's attribute length
    
class Array:
    size = ArraySize()  # define descriptor object
    
    def __init__(self, arr):
        self.arr = arr
        
array_obj = Array([1, 2, 3])

In [22]:
array_obj.size

3

we might access the attribute of the object this way:

In [4]:
print(len(array_obj.arr))

3


...or, we can make this much shorter:

In [12]:
array_obj.size

3

In [13]:
array_obj.arr.append(5)

In [14]:
array_obj.size

4

we could rewrite this task with ***property***:

In [15]:
class Array:
    def __init__(self, arr):
        self.arr = arr
        
    @property
    def size(self):
        return len(self.arr)
    
array_obj = Array([1, 2, 3])
print(array_obj.size)

3


***properties*** are created to simplify the code of ***descriptors***

any ***property*** can be expressed as a ***descriptor*** but not any ***descriptor*** can be expressed as a ***property***

In [19]:
class Temperature:
    # getter
    def __get__(self, instance, owner):
        print("get temperature")
        return instance._temperature  # 4. return 'validated' by 'setter' value
    
    # setter
    def __set__(self, instance, value):  # 2. takes the instance of class (GlassOfWater) and value (98)
        print("set temperature")
        if not (0 <= value <= 100):
            print("Temperature must be in range 0...100!")
            return
        instance._temperature = value  # 3. after validation, the 'value' is set to "instance" under yet protected attribute
        
class GlassOfWater:
    temperature = Temperature()
    
    def __init__(self, temp):
        self.temperature = temp  # 1. by instantiating it uses "descriptor's setter" to vaidate and assign the value of "temp"
        
    def heat(self):
        self.temperature += 1  # calls both getter and setter

In [22]:
glass = GlassOfWater(98)

set temperature


In [23]:
type(glass).__dict__

mappingproxy({'__module__': '__main__',
              'temperature': <__main__.Temperature at 0x161265efc10>,
              '__init__': <function __main__.GlassOfWater.__init__(self, temp)>,
              'heat': <function __main__.GlassOfWater.heat(self)>,
              '__dict__': <attribute '__dict__' of 'GlassOfWater' objects>,
              '__weakref__': <attribute '__weakref__' of 'GlassOfWater' objects>,
              '__doc__': None})

In [4]:
glass.__dict__

{'_temperature': 98}

In [5]:
glass.heat()
glass.heat()
glass.heat()

get temperature
set temperature
get temperature
set temperature
get temperature
set temperature
Temperature must be in range 0...100!


In [6]:
glass.temperature

get temperature


100

In [7]:
glass.__dict__

{'_temperature': 100}

### Descriptors Advanced

In [2]:
class Temperature:
    def __init__(self, min_temp, max_temp):  # add custom parameters for min-max range
        self.min_temp = min_temp
        self.max_temp = max_temp

    def __get__(self, instance, owner):
        print("get temperature")
        return instance._temperature

    def __set__(self, instance, value):
        print("set temperature")
        if not (self.min_temp <= value <= self.max_temp):
            print(f"Temperature must be in range {self.min_temp}...{self.max_temp}!")
            return
        instance._temperature = value
        
class GlassOfWater:
    temperature = Temperature(10, 30)  # define min-max temperature range
    
    def __init__(self, temp):
        self.temperature = temp
        
    def heat(self):
        self.temperature += 1

In [3]:
glass = GlassOfWater(29)

set temperature


In [4]:
vars(glass)

{'_temperature': 29}

In [5]:
vars(type(glass))

mappingproxy({'__module__': '__main__',
              'temperature': <__main__.Temperature at 0x1f4cd5e49d0>,
              '__init__': <function __main__.GlassOfWater.__init__(self, temp)>,
              'heat': <function __main__.GlassOfWater.heat(self)>,
              '__dict__': <attribute '__dict__' of 'GlassOfWater' objects>,
              '__weakref__': <attribute '__weakref__' of 'GlassOfWater' objects>,
              '__doc__': None})

In [8]:
vars(vars(type(glass))["temperature"])

{'min_temp': 10, 'max_temp': 30}

In [9]:
glass.heat()

get temperature
set temperature


In [10]:
glass.temperature

get temperature


30

In [11]:
glass.heat()

get temperature
set temperature
Temperature must be in range 10...30!


#### working with more than one ***data descriptors*** in one class

In [23]:
class Temperature:
    def __init__(self, min_temp, max_temp):
        self.min_temp = min_temp
        self.max_temp = max_temp
    
    def __get__(self, instance, owner):
        print("get temperature")
        return instance._temperature
    
    def __set__(self, instance, value):
        print("set temperature", value)
        if not (self.min_temp <= value <= self.max_temp):
            print(f"Temperature must be in range {self.min_temp}...{self.max_temp}!")
            return
        instance._temperature = value
        
class GlassOfWater:
    water_temperature = Temperature(0, 100)
    air_temperature = Temperature(10, 30)  # add air temperature descriptor
    
    def __init__(self, water_temp, air_temp):
        self.water_temperature = water_temp
        self.air_temperature = air_temp
        
    def water_heat(self):
        self.water_temperature += 1
        
    def air_heat(self):
        self.air_temperature += 1

In [24]:
glass = GlassOfWater(29, 20)

set temperature 29
set temperature 20


In [19]:
glass.__dict__

{'_temperature': 20}

In [20]:
print("Water: ", glass.water_temperature)
print("Air: ", glass.air_temperature)

get temperature
Water:  20
get temperature
Air:  20


we got wrong behavior, since second attribute `self.air_temperature` has overwritten `self.water_temperature`. let dig into it:

it's happening inside initilizer of the `GlassOfWater` first, then we go to descriptor and since it calls two times the ***same*** `__set__` it overwrites the value of the first descriptor `water_temperature` by `air_temperature`

In [30]:
class Temperature:
    def __init__(self, min_temp, max_temp):
        self.min_temp = min_temp
        self.max_temp = max_temp
        
    def __set_name__(self, owner, name):  # name of the object the descripor attached to
        print("class owner:", owner, "instance name: ", name)
        # by instantiating the descriptor, the 'public' and 'protected' attribute names are being created
        self.public_name = name
        self.protected_name = "_" + name
    
    def __get__(self, instance, owner):
        print("get temperature")
        return instance._temperature
    
    def __set__(self, instance, value):
        print("set temperature", value)
        if not (self.min_temp <= value <= self.max_temp):
            print(f"Temperature must be in range {self.min_temp}...{self.max_temp}!")
            return
        instance._temperature = value
        
class GlassOfWater:
    water_temperature = Temperature(0, 100)
    air_temperature = Temperature(10, 30)
    
    def __init__(self, water_temp, air_temp):
        self.water_temperature = water_temp
        self.air_temperature = air_temp
        
    def water_heat(self):
        self.water_temperature += 1
        
    def air_heat(self):
        self.air_temperature += 1

class owner: <class '__main__.GlassOfWater'> instance name:  water_temperature
class owner: <class '__main__.GlassOfWater'> instance name:  air_temperature


In [31]:
glass = GlassOfWater(29, 20)

set temperature 29
set temperature 20


In [32]:
# object dict
glass.__dict__

{'_temperature': 20}

In [33]:
# object's type dict
type(glass).__dict__

mappingproxy({'__module__': '__main__',
              'water_temperature': <__main__.Temperature at 0x1c91cc9aed0>,
              'air_temperature': <__main__.Temperature at 0x1c91cc9a490>,
              '__init__': <function __main__.GlassOfWater.__init__(self, water_temp, air_temp)>,
              'water_heat': <function __main__.GlassOfWater.water_heat(self)>,
              'air_heat': <function __main__.GlassOfWater.air_heat(self)>,
              '__dict__': <attribute '__dict__' of 'GlassOfWater' objects>,
              '__weakref__': <attribute '__weakref__' of 'GlassOfWater' objects>,
              '__doc__': None})

In [34]:
# object's type water_temperature attribute(descriptor)
type(glass).__dict__['water_temperature'].__dict__

{'min_temp': 0,
 'max_temp': 100,
 'public_name': 'water_temperature',
 'protected_name': '_water_temperature'}

In [35]:
# object's type air_temperature attribute(descriptor)
type(glass).__dict__['air_temperature'].__dict__

{'min_temp': 10,
 'max_temp': 30,
 'public_name': 'air_temperature',
 'protected_name': '_air_temperature'}

with descriptor's `__set_name__` method, we have created instance ***public*** and ***protected*** attibute names, based on the descriptor's variable name

we can use these distinct attribute names, defined in the descriptor, for defining the attibute names for the class instance:

In [8]:
class Temperature:
    def __init__(self, min_temp, max_temp):
        self.min_temp = min_temp
        self.max_temp = max_temp
        
    def __set_name__(self, owner, name):
        print("__set_name__ called", name)
        self.public_name = name
        self.protected_name = "_" + name
    
    def __get__(self, instance, owner):
        print(f"get {self.public_name}")
        return getattr(instance, self.protected_name)
    
    def __set__(self, instance, value):
        print(f"set {self.public_name}", value)
        if not (self.min_temp <= value <= self.max_temp):
            print(f"Temperature must be in range {self.min_temp}...{self.max_temp}!")
            return
        setattr(instance, self.protected_name, value)
        
class GlassOfWater:
    water_temperature = Temperature(0, 100)
    air_temperature = Temperature(10, 30)
    
    def __init__(self, water_temp, air_temp):
        self.water_temperature = water_temp
        self.air_temperature = air_temp
        
    def water_heat(self):
        self.water_temperature += 1
        
    def air_heat(self):
        self.air_temperature += 1

__set_name__ called water_temperature
__set_name__ called air_temperature


In [9]:
GlassOfWater.__dict__

mappingproxy({'__module__': '__main__',
              'water_temperature': <__main__.Temperature at 0x19728fd7210>,
              'air_temperature': <__main__.Temperature at 0x19728fd7390>,
              '__init__': <function __main__.GlassOfWater.__init__(self, water_temp, air_temp)>,
              'water_heat': <function __main__.GlassOfWater.water_heat(self)>,
              'air_heat': <function __main__.GlassOfWater.air_heat(self)>,
              '__dict__': <attribute '__dict__' of 'GlassOfWater' objects>,
              '__weakref__': <attribute '__weakref__' of 'GlassOfWater' objects>,
              '__doc__': None})

In [10]:
GlassOfWater.__dict__["water_temperature"].__dict__

{'min_temp': 0,
 'max_temp': 100,
 'public_name': 'water_temperature',
 'protected_name': '_water_temperature'}

In [11]:
GlassOfWater.__dict__["air_temperature"].__dict__

{'min_temp': 10,
 'max_temp': 30,
 'public_name': 'air_temperature',
 'protected_name': '_air_temperature'}

In [12]:
glass = GlassOfWater(29, 20)

set water_temperature 29
set air_temperature 20


In [13]:
print("Water: ", glass.water_temperature)
print("Air: ", glass.air_temperature)

get water_temperature
Water:  29
get air_temperature
Air:  20


In [14]:
glass.__dict__

{'_water_temperature': 29, '_air_temperature': 20}

In [15]:
glass.water_heat()
glass.air_heat()

get water_temperature
set water_temperature 30
get air_temperature
set air_temperature 21


having defined `__set_name__` each instance of the same descriptor in one class is being written with its own attibute name based on the descriptor variable name, and then `__set__` is used with correct name for each descriptor.